# Créer des applications de génération d'images (Azure OpenAI)

Les LLM ne se limitent pas à la génération de texte. Vous pouvez aussi générer des images à partir de descriptions textuelles. Les images en tant que modalité sont utiles dans les domaines de la technologie médicale, de l'architecture, du tourisme, du développement de jeux, du marketing, et plus encore. Dans cette leçon, nous travaillons avec les modèles **GPT Image** actuels sur Microsoft Foundry.

## Objectifs d'apprentissage

À la fin de cette leçon, vous serez capables de :

- Expliquer ce qu'est la génération d'images et où elle est utile.
- Comprendre la famille de modèles `gpt-image` et en quoi elle diffère des modèles DALL·E hérités.
- Construire une application de génération d'images et éditer des images.

## Qu'est-ce que la génération d'images ?

Les modèles de génération d'images créent des images à partir d'une invite textuelle. Les modèles modernes tels que `gpt-image` apprennent la relation entre le texte et les images lors de l'entraînement, puis transforment itérativement un bruit aléatoire en une image correspondant à votre description.

Deux familles de modèles d'images bien connues sont :

- **`gpt-image` (OpenAI)** — la génération actuelle utilisée dans cette leçon. Il prend en charge la génération texte-vers-image et l'édition d'images (retouche avec un masque).
- **Midjourney** — un modèle tiers populaire avec son propre service et un flux de travail basé sur Discord.

> Les anciens modèles d’images OpenAI — **DALL·E 2** et **DALL·E 3** — sont hérités. DALL·E 3 n’est plus disponible pour de nouveaux déploiements, et des fonctionnalités comme `create_variation` existaient uniquement dans DALL·E 2. Utilisez les modèles `gpt-image` pour les nouvelles applications.

Sur Microsoft Foundry, **`gpt-image-2`** est le modèle d’image le plus récent et le plus performant, et est celui recommandé par défaut. `gpt-image-1.5` et `gpt-image-1-mini` sont aussi généralement disponibles.

> **Important :** les modèles `gpt-image` renvoient l’image générée en **base64** (`b64_json`), et non sous forme d'URL. Votre code décode la chaîne base64 en octets puis la sauvegarde — aucune URL d’image n’est disponible pour téléchargement.


## Construire votre première application de génération d'image

Que faut-il pour construire une application de génération d'images ? Vous avez besoin des bibliothèques suivantes :

- **python-dotenv**, il est fortement recommandé d'utiliser cette bibliothèque pour garder vos secrets dans un fichier *.env* séparé du code.
- **openai**, cette bibliothèque est ce que vous utiliserez pour interagir avec l'API OpenAI.
- **pillow**, pour travailler avec des images en Python.

Si ce n'est pas encore fait, suivez les instructions sur la page [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) pour créer une ressource et un modèle Microsoft Foundry. Sélectionnez **gpt-image-2** comme modèle (le dernier modèle d'image Azure OpenAI ; DALL·E est obsolète).

1. Créez un fichier *.env* avec le contenu suivant :

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Trouvez ces informations dans le portail Microsoft Foundry pour votre ressource dans la section "Deployments".



1. Rassemblez les bibliothèques ci-dessus dans un fichier nommé *requirements.txt* comme suit :

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Ensuite, créez un environnement virtuel et installez les bibliothèques :


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Pour Windows, utilisez les commandes suivantes pour créer et activer votre environnement virtuel :

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Ajoutez le code suivant dans un fichier appelé *app.py* :

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # importer dotenv
    dotenv.load_dotenv()

    # configurer le client Azure OpenAI (Microsoft Foundry).
    # Les modèles d'image nécessitent une version récente de l'API — consultez la documentation Foundry pour celle requise par votre modèle.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Créer une image en utilisant l'API de génération d'images
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Entrez votre texte d'invite ici
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # par exemple "gpt-image-2"
        )
        # Définir le répertoire pour l'image stockée
        image_dir = os.path.join(os.curdir, 'images')

        # Si le répertoire n'existe pas, le créer
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialiser le chemin de l'image (notez que le type de fichier doit être png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # Les modèles gpt-image retournent l'image au format base64 (b64_json), pas une URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Afficher l'image dans le visionneur d'images par défaut
        image = Image.open(image_path)
        image.show()

    # gérer les exceptions
    except BadRequestError as err:
        print(err)

    ```

Expliquons ce code :

- Tout d'abord, nous importons les bibliothèques dont nous avons besoin, y compris la bibliothèque OpenAI, la bibliothèque dotenv, le module base64, et la bibliothèque Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Ensuite, nous chargeons les variables d'environnement depuis le fichier *.env*.

    ```python
    # importer dotenv
    dotenv.load_dotenv()
    ```

- Après cela, nous configurons le client Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Ensuite, nous générons l'image :

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Entrez le texte de votre invite ici
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    Les modèles `gpt-image` renvoient l'image sous forme de chaîne **base64** dans `data[0].b64_json`. Nous la décodons en octets et l'écrivons dans un fichier — il n'y a pas d'URL pour le téléchargement.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Enfin, nous ouvrons l'image et utilisons le visualiseur d'images standard pour l'afficher :

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Plus de détails sur la génération de l'image

Regardons les paramètres de `images.generate` :

- **prompt**, est le texte utilisé pour générer l'image. Ici c'est "Lapin sur un cheval, tenant une sucette, dans une prairie brumeuse où poussent des jonquilles".
- **size**, est la taille de l'image générée (par exemple `1024x1024`, `1536x1024`, `1024x1536`, ou `"auto"`).
- **n**, est le nombre d'images générées. Ici nous en générons une.
- **model**, est le nom de votre déploiement du modèle d'image (par exemple `gpt-image-2`).

> Les modèles d'images ne prennent pas de paramètre `temperature` — c'est un contrôle pour la génération de texte. Pour obtenir de la variété, appelez l'API à nouveau ; pour réduire la variété, rendez votre prompt plus spécifique.

## Capacités supplémentaires de la génération d'image

Vous avez vu comment générer une image en quelques lignes de Python. Les modèles `gpt-image` peuvent aussi **éditer** une image existante. En fournissant une image, un **masque** optionnel (qui indique la zone à modifier), et un prompt, vous pouvez modifier une partie d'une image — par exemple, ajouter un chapeau à notre lapin.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# les modifications sont également retournées en base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

L'image de base contient juste le lapin ; l'image finale a le chapeau sur le lapin.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :
Ce document a été traduit à l'aide du service de traduction automatique [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforçions d'assurer l'exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue native doit être considéré comme la source faisant autorité. Pour les informations critiques, il est recommandé de recourir à une traduction professionnelle réalisée par un humain. Nous ne saurions être tenus responsables des malentendus ou erreurs d'interprétation découlant de l'utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
